In [1]:
import json
import pickle
import pandas as pd
from datetime import datetime
from imblearn.over_sampling import RandomOverSampler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
class ModelTrainer:

    def __init__(self, model_id, model_type, base_params):
        self.model_id = model_id
        self.model_type = model_type
        self.params = self._clean_and_convert_params(base_params)
        
    def _clean_and_convert_params(self, raw_params):
        # remove statistics irrelevant to model training
        useless_keys = ["achieved_value"]
        clean_params = {k: v for k, v in raw_params.items() if k not in useless_keys}
        
        # set estimators
        clean_params["n_estimators"] = 50
        
        # choose hyperparameters taht need to be int
        int_keys = ["max_depth", "min_samples_leaf", "min_samples_split", "min_child_weight"]
        for key in int_keys:
            if key in clean_params:
                try:
                    clean_params[key] = int(clean_params[key])
                except (ValueError, TypeError):
                    pass
        return clean_params

    def run_training(self):
        
        print(f"Task: {self.model_id} - {self.model_type}")
        
        # load training data
        features_path = f"data/X_train_{self.model_id}.csv"
        label_path = f"data/y_train_{self.model_id}.csv"
        X = pd.read_csv(features_path, keep_default_na=False)
        y = pd.read_csv(label_path, keep_default_na=False).values.ravel()

        # choose model set which need oversampling
        if self.model_id[-1] in ['a', 'b']:
            sampler = RandomOverSampler(random_state=2024)
            X_final, y_final = sampler.fit_resample(X, y)
        else:
            X_final, y_final = X, y

        # choose classifier
        if self.model_type == "xgboost":
            classifier = XGBClassifier(**self.params)
        else:
            classifier = RandomForestClassifier(**self.params)

        # fit the model and save result
        classifier.fit(X_final, y_final)
        save_path = f"models/{self.model_id}_{self.model_type}.pkl"
        with open(save_path, "wb") as f:
            pickle.dump(classifier, f)

In [3]:
def main():
    start_time = datetime.now()
    
    # set model id and type
    model_list = ["model_1", "model_2", "model_1a", "model_2a", "model_1b", "model_2b"]
    model_type_list = ["xgboost", "rf"]

    # train the model
    for m_id in model_list:
        for m_type in model_type_list:
            # set hyperparameter
            param_file = f'results/{m_id}_{m_type}_best_within_one.json'
            with open(param_file, 'r', encoding='utf-8') as f:
                best_params = json.load(f)
            
            # train model
            trainer = ModelTrainer(model_id=m_id, model_type=m_type, base_params=best_params)
            trainer.run_training()

    print(f"Time: {datetime.now() - start_time}")

if __name__ == "__main__":
    main()

Task: model_1 - xgboost
Task: model_1 - rf
Task: model_2 - xgboost
Task: model_2 - rf
Task: model_1a - xgboost
Task: model_1a - rf
Task: model_2a - xgboost
Task: model_2a - rf
Task: model_1b - xgboost
Task: model_1b - rf
Task: model_2b - xgboost
Task: model_2b - rf
Time: 0:00:15.051476
